In [1]:
!pip install torch transformers accelerate
!pip install mamba-ssm causal-conv1d>=1.1.0

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class MambaSummarizer:
    def __init__(self, model_name="state-spaces/mamba-130m-hf"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,  # Экономия памяти
            device_map="auto"
        )
        self.model.eval()

    def get_sentence_embeddings(self, sentences):
        """Получаем эмбеддинги предложений через скрытые состояния Mamba"""
        embeddings = []

        for sent in sentences:
            # Токенизируем
            inputs = self.tokenizer(sent, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

            with torch.no_grad():
                # Получаем выходы. Mamba возвращает logits и скрытые состояния
                outputs = self.model(**inputs, output_hidden_states=True)

                # Берем последнее скрытое состояние (последний токен) как эмбеддинг предложения
                # У Mamba hidden_states - это tuple из (последнего слоя)
                last_hidden = outputs.hidden_states[-1]  # [batch, seq_len, hidden_dim]
                # Усредняем по токенам или берем [:, -1, :] — пробуйте оба варианта
                sent_emb = last_hidden.mean(dim=1).squeeze().cpu().numpy()
                embeddings.append(sent_emb)

        return np.array(embeddings)

    def extractive_summarize(self, text, ratio=0.3):
        """
        Экстрактивная суммаризация: выбираем топ N предложений по схожести с центром документа.
        """
        # Разбиваем на предложения (примитивно, лучше использовать nltk или spaCy)
        sentences = [s.strip() for s in text.split('. ') if len(s.strip()) > 20]

        if len(sentences) <= 3:
            return text

        # Получаем эмбеддинги
        embeds = self.get_sentence_embeddings(sentences)

        # Вычисляем центроид документа (среднее всех эмбеддингов)
        centroid = np.mean(embeds, axis=0)

        # Считаем косинусное сходство каждого предложения с центроидом
        scores = cosine_similarity(embeds, centroid.reshape(1, -1)).flatten()

        # Выбираем топ N предложений
        num_sentences = max(1, int(len(sentences) * ratio))
        top_indices = np.argsort(scores)[-num_sentences:]
        top_indices = sorted(top_indices)  # Сохраняем порядок в тексте

        return '. '.join([sentences[i] for i in top_indices])

# Использование
summarizer = MambaSummarizer()
text = "Ваш длинный текст. Здесь много предложений. Mamba отлично справляется с длинными контекстами."
summary = summarizer.extractive_summarize(text)
print(summary)

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.79k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/517M [00:00<?, ?B/s]

[transformers] The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.


Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Ваш длинный текст. Здесь много предложений. Mamba отлично справляется с длинными контекстами.


In [3]:
summarizer = MambaSummarizer()
text = 'В последнее десятилетие человечество стало свидетелем беспрецедентного ускорения технологического прогресса. Если раньше научные открытия происходили с интервалом в десятилетия, то сегодня новые прорывные идеи возникают ежемесячно, а иногда и еженедельно. Особенно ярко этот тренд проявляется в двух, казалось бы, различных областях: квантовой физике и искусственном интеллекте. Однако, как показывают последние исследования, эти дисциплины не просто развиваются параллельно, но и начинают активно пересекаться, создавая эффект синергии, который может кардинально изменить наше представление о возможностях вычислительной техники.Квантовые компьютеры, ещё недавно считавшиеся чисто теоретическим конструктом, сегодня уже существуют в виде прототипов, способных решать задачи, недоступные для классических суперкомпьютеров. Речь идёт не о простом ускорении вычислений, а о принципиально иной парадигме обработки информации. Если классический бит может находиться в состоянии либо нуля, либо единицы, то кубит благодаря принципу суперпозиции способен быть одновременно и нулём, и единицей с определённой вероятностью. Это позволяет квантовому компьютеру перебирать огромное количество вариантов решений за один такт времени, что особенно важно для задач оптимизации, криптографии и моделирования сложных молекулярных структур. Фармацевтические компании уже сегодня инвестируют миллиарды долларов в разработку квантовых алгоритмов для моделирования взаимодействия лекарственных препаратов с белками, что может сократить сроки разработки новых лекарств с десяти лет до нескольких месяцев.Однако существует серьёзная проблема, сдерживающая массовое внедрение квантовых вычислений. Речь идёт о декогеренции — потере квантовых свойств при взаимодействии с окружающей средой. Даже незначительная флуктуация температуры или электромагнитное излучение способны разрушить хрупкое состояние кубита, что приводит к ошибкам в вычислениях. Именно здесь на помощь приходят нейросетевые технологии. Исследователи из Google и IBM активно экспериментируют с глубокими нейронными сетями, которые способны предсказывать и компенсировать ошибки декогеренции в реальном времени. Нейросеть анализирует шумовые паттерны и подстраивает параметры управления кубитами так, чтобы максимально продлить время когерентности. Этот симбиоз квантовой физики и машинного обучения получил название квантовое машинное обучение, и он считается одним из самых многообещающих направлений науки XXI века.Параллельно с этим развиваются и сами нейросети. Современные большие языковые модели, такие как GPT-4 и её аналоги, содержат триллионы параметров и обучаются на терабайтах текстовых данных. Однако их архитектура, основанная на трансформерах, сталкивается с фундаментальным ограничением: механизм внимания требует квадратичных вычислительных затрат при увеличении длины контекста. Если документ содержит 100 тысяч слов, трансформеру необходимо обработать 10 миллиардов попарных взаимодействий между токенами, что делает обработку сверхдлинных текстов крайне неэффективной. Именно в этот момент на сцену выходят альтернативные архитектуры, такие как Mamba, основанная на моделях пространства состояний. Mamba демонстрирует линейную масштабируемость, что позволяет ей обрабатывать контексты длиной до миллиона токенов, не теряя при этом в качестве понимания взаимосвязей между удалёнными частями текста. Это открывает новые горизонты для анализа многотомных научных работ, юридических документов и даже целых библиотек.Социальные последствия этих технологических сдвигов не менее значимы, чем их технические аспекты. Эксперты предупреждают, что появление искусственного общего интеллекта (AGI) может произойти уже в течение ближайших пяти-десяти лет, и это ставит перед человечеством сложнейшие этические вопросы. Кто будет нести ответственность за действия автономных систем, принимающих решения в медицине, судопроизводстве и военной сфере? Как гарантировать, что алгоритмы не будут дискриминировать определённые социальные группы, если их обучающие данные содержат исторические предубеждения? Эти проблемы уже сейчас обсуждаются на международных саммитах по искусственному интеллекту, однако единой регуляторной рамки пока не выработано. Европейский союз предложил один из первых законопроектов — AI Act, который классифицирует риски и вводит требования к прозрачности систем, но многие эксперты считают этот подход излишне бюрократическим и тормозящим инновации.Не менее важным аспектом является энергопотребление. Центры обработки данных, на которых обучаются современные нейросети, потребляют столько же электроэнергии, сколько небольшие государства. Например, обучение одной модели уровня GPT-3 требует около 1300 мегаватт-часов, что сопоставимо с годовым потреблением энергии ста среднестатистических домохозяйств. Американское министерство энергетики прогнозирует, что к 2030 году доля ИИ-вычислений в общем энергопотреблении планеты достигнет 3-5 процентов, что сопоставимо с авиационной отраслью. В связи с этим учёные активно ищут новые подходы к энергоэффективному обучению, включая использование квантовых нейросетей, которые потенциально могут выполнять те же вычисления с на порядок меньшим энергопотреблением.Россия также не остаётся в стороне от этих процессов. В стране реализуется дорожная карта по развитию квантовых технологий до 2030 года, в рамках которой уже созданы первые отечественные кубиты на основе сверхпроводников и нейтральных атомов. Однако, по мнению экспертов РАН, основная проблема заключается не столько в создании аппаратной базы, сколько в подготовке квалифицированных кадров. На всю страну приходится менее тысячи специалистов, которые одновременно разбираются в квантовой физике и машинном обучении, тогда как для полноценного развития отрасли требуется как минимум в десять раз больше. Система образования пока не успевает за стремительными изменениями в промышленности: университетские программы обновляются раз в пять лет, а технологии меняются каждые полгода. В результате возникает разрыв между академической наукой и реальным сектором, который некоторые предприниматели пытаются преодолеть через корпоративные университеты и онлайн-платформы.Следующим этапом эволюции, вероятно, станет так называемый вычисления на биологических системах. Уже сейчас ведутся эксперименты по использованию нейронов в культуре для решения простейших логических задач, а стартап Neuralink демонстрирует прототипы интерфейсов мозг-компьютер, позволяющие парализованным людям управлять курсором силой мысли. Если объединить эти разработки с квантовыми вычислениями и нейросетями, то мы получим гибридную систему, в которой биологические, квантовые и искусственные компоненты будут работать как единый организм. Некоторые футурологи, в частности Рэй Курцвейл, предсказывают, что к 2045 году такие системы достигнут сингулярности — момента, когда интеллект машин превзойдёт человеческий и начнёт самоускоряться, создавая всё более совершенные версии самого себя.Однако не все разделяют этот оптимистичный взгляд. Критики указывают на то, что сложность биологического мозга до сих пор полностью не изучена: мы не знаем, как именно возникает сознание, что такое субъективный опыт и каковы физические корреляты мышления. Без понимания этих фундаментальных основ любые разговоры о цифровой копии сознания остаются спекуляцией, далёкой от практической реализации. Более того, даже если технические проблемы будут решены, останутся неразрешёнными философские вопросы: имеет ли право человек загружать своё сознание в компьютер, и будет ли это по-прежнему "он сам" или лишь его имитация, неспособная к подлинным чувствам и переживаниям? Эти дилеммы уже сейчас становятся предметом ожесточённых дискуссий между трансгуманистами и гуманистами, и вряд ли они найдут консенсус в обозримом будущем.В то же время нельзя игнорировать и экономическую составляющую. Технологические гиганты, такие как Microsoft, Amazon и Google, уже вложили в квантовые и ИИ-проекты более 100 миллиардов долларов, и эта цифра продолжает расти с каждым годом. Аналитики McKinsey прогнозируют, что рынок квантовых вычислений достигнет 2 триллионов долларов к 2035 году, создав миллионы новых рабочих мест в таких сферах, как квантовое программирование, криптографическая безопасность и материаловедение. Одновременно с этим исчезнут сотни традиционных профессий: бухгалтеры, переводчики, операторы колл-центров и даже некоторые категории юристов будут вытеснены алгоритмами, способными выполнять их работу быстрее и без ошибок. Это неизбежно приведёт к социальной напряжённости, что требует от правительств заблаговременной разработки программ переквалификации и базового безусловного дохода.Подводя итог, можно констатировать, что человечество стоит на пороге величайшей трансформации со времён промышленной революции. В отличие от прошлых эпох, где изменения растягивались на столетия, сегодня технологический сдвиг происходит в пределах жизни одного поколения. Наши дети будут жить в мире, где квантовые компьютеры будут такими же обыденными, как сегодня смартфоны, а нейросети станут неотъемлемой частью образования, здравоохранения и управления государством. Удастся ли нам использовать этот шанс во благо, или мы повторим ошибки предыдущих технологических революций, где выгоду получали лишь избранные, — вопрос открытый. Ясно одно: медлить с адаптацией больше нельзя, и каждый из нас, будь то учёный, предприниматель или просто гражданин, несёт ответственность за то, каким будет этот новый мир. Время пассивного наблюдения прошло, наступила эпоха осознанного участия, и от нашей коллективной мудрости зависит, станет ли будущее светлым или мы столкнёмся с новыми, непредвиденными угрозами.'
summary = summarizer.extractive_summarize(text)
print(summary)

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

Однако, как показывают последние исследования, эти дисциплины не просто развиваются параллельно, но и начинают активно пересекаться, создавая эффект синергии, который может кардинально изменить наше представление о возможностях вычислительной техники.Квантовые компьютеры, ещё недавно считавшиеся чисто теоретическим конструктом, сегодня уже существуют в виде прототипов, способных решать задачи, недоступные для классических суперкомпьютеров. Фармацевтические компании уже сегодня инвестируют миллиарды долларов в разработку квантовых алгоритмов для моделирования взаимодействия лекарственных препаратов с белками, что может сократить сроки разработки новых лекарств с десяти лет до нескольких месяцев.Однако существует серьёзная проблема, сдерживающая массовое внедрение квантовых вычислений. Исследователи из Google и IBM активно экспериментируют с глубокими нейронными сетями, которые способны предсказывать и компенсировать ошибки декогеренции в реальном времени. Нейросеть анализирует шумовые п

In [4]:
o = 'Однако, как показывают последние исследования, эти дисциплины не просто развиваются параллельно, но и начинают активно пересекаться, создавая эффект синергии, который может кардинально изменить наше представление о возможностях вычислительной техники.Квантовые компьютеры, ещё недавно считавшиеся чисто теоретическим конструктом, сегодня уже существуют в виде прототипов, способных решать задачи, недоступные для классических суперкомпьютеров. Фармацевтические компании уже сегодня инвестируют миллиарды долларов в разработку квантовых алгоритмов для моделирования взаимодействия лекарственных препаратов с белками, что может сократить сроки разработки новых лекарств с десяти лет до нескольких месяцев.Однако существует серьёзная проблема, сдерживающая массовое внедрение квантовых вычислений. Исследователи из Google и IBM активно экспериментируют с глубокими нейронными сетями, которые способны предсказывать и компенсировать ошибки декогеренции в реальном времени. Нейросеть анализирует шумовые паттерны и подстраивает параметры управления кубитами так, чтобы максимально продлить время когерентности. Однако их архитектура, основанная на трансформерах, сталкивается с фундаментальным ограничением: механизм внимания требует квадратичных вычислительных затрат при увеличении длины контекста. Mamba демонстрирует линейную масштабируемость, что позволяет ей обрабатывать контексты длиной до миллиона токенов, не теряя при этом в качестве понимания взаимосвязей между удалёнными частями текста. В связи с этим учёные активно ищут новые подходы к энергоэффективному обучению, включая использование квантовых нейросетей, которые потенциально могут выполнять те же вычисления с на порядок меньшим энергопотреблением.Россия также не остаётся в стороне от этих процессов. На всю страну приходится менее тысячи специалистов, которые одновременно разбираются в квантовой физике и машинном обучении, тогда как для полноценного развития отрасли требуется как минимум в десять раз больше. Система образования пока не успевает за стремительными изменениями в промышленности: университетские программы обновляются раз в пять лет, а технологии меняются каждые полгода. В результате возникает разрыв между академической наукой и реальным сектором, который некоторые предприниматели пытаются преодолеть через корпоративные университеты и онлайн-платформы.Следующим этапом эволюции, вероятно, станет так называемый вычисления на биологических системах. Уже сейчас ведутся эксперименты по использованию нейронов в культуре для решения простейших логических задач, а стартап Neuralink демонстрирует прототипы интерфейсов мозг-компьютер, позволяющие парализованным людям управлять курсором силой мысли. Некоторые футурологи, в частности Рэй Курцвейл, предсказывают, что к 2045 году такие системы достигнут сингулярности — момента, когда интеллект машин превзойдёт человеческий и начнёт самоускоряться, создавая всё более совершенные версии самого себя.Однако не все разделяют этот оптимистичный взгляд. Более того, даже если технические проблемы будут решены, останутся неразрешёнными философские вопросы: имеет ли право человек загружать своё сознание в компьютер, и будет ли это по-прежнему "он сам" или лишь его имитация, неспособная к подлинным чувствам и переживаниям? Эти дилеммы уже сейчас становятся предметом ожесточённых дискуссий между трансгуманистами и гуманистами, и вряд ли они найдут консенсус в обозримом будущем.В то же время нельзя игнорировать и экономическую составляющую. Удастся ли нам использовать этот шанс во благо, или мы повторим ошибки предыдущих технологических революций, где выгоду получали лишь избранные, — вопрос открытый'
print(len(o))
print(len(text))

3638
9646
